# Exploratory Data Analysis (EDA) - LeFoodSet Leftovers Dataset
**LeFoodSet Leftovers Food** Analisis ini meliputi statistik dasar, distribusi data, analisis gambar, serta hubungan antara sisa makanan dengan penilaian visual.

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/ml-food-waste-estimation'  # adjust if needed
    os.chdir(PROJECT_ROOT)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    PROJECT_ROOT = os.getcwd()

sys.path.insert(0, 'src')
print(f'{"Colab" if IN_COLAB else "Local"} | cwd: {os.getcwd()}')

## 1. Import Library
Kita mengimpor library yang diperlukan untuk analisis data, visualisasi, dan pemrosesan gambar.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import numpy as np
from scipy.stats import pearsonr

sns.set_theme(style='whitegrid')

## 2. Load Metadata
Kita membaca file metadata `.xlsx` yang berisi informasi umum tentang setiap sampel makanan, lalu menghitung kolom baru `Weight Leftover (g)` yang merupakan selisih antara berat sebelum dan sesudah dimakan.

In [ ]:
metadata_path = os.path.join('data', 'data_original.xlsx')
df = pd.read_excel(metadata_path)

df['Weight Leftover (g)'] = df['Weight Before Eaten (g)'] - df['Weight After Eaten (g)']

print('=== Informasi Metadata (semua baris) ===')
print('Jumlah data:', df.shape[0])
print('Jumlah kolom:', df.shape[1])
print('Kolom:', df.columns.tolist())

# Filter to usable samples (those with segmented images on disk) -- same logic as training
SEG_BEFORE_DIR = os.path.join('data', 'segmented', 'data_before')
SEG_AFTER_DIR  = os.path.join('data', 'segmented', 'data_after')

def _seg_filename(raw):
    return f"{raw[:3]}_{raw}"

available_bef = {f for _, _, files in os.walk(SEG_BEFORE_DIR) for f in files}
available_aft = {f for _, _, files in os.walk(SEG_AFTER_DIR)  for f in files}
mask = (
    df['Image Before Eaten'].apply(_seg_filename).isin(available_bef) &
    df['Image After Eaten'].apply(_seg_filename).isin(available_aft)
)
df_usable = df[mask].reset_index(drop=True).copy()
df_usable['consumption_ratio'] = (
    df_usable['Weight After Eaten (g)'] / df_usable['Weight Before Eaten (g)']
).clip(0.0, 1.0)

print(f'\nBaris total          : {len(df)}')
print(f'Baris tanpa segmented: {(~mask).sum()}  (dilewati saat training)')
print(f'Baris usable         : {len(df_usable)}  (digunakan saat training)')
print(f'Kategori makanan     : {df_usable["Name of the food"].nunique()}')
df_usable.head()

## 3. EDA Metadata
Kita menganalisis distribusi jenis makanan, berat sebelum & sesudah makan, sisa makanan, dan distribusi skor penilaian visual.

In [ ]:
# Top 10 jenis makanan (usable samples)
food_counts = df_usable['Name of the food'].value_counts().head(10)
food_top = food_counts.reset_index()
food_top.columns = ['food', 'count']
plt.figure(figsize=(10, 6))
sns.barplot(data=food_top, x='count', y='food', hue='food', palette='mako', legend=False)
plt.title('Top 10 Jenis Makanan (524 usable samples)')
plt.xlabel('Jumlah')
plt.ylabel('Jenis Makanan')
plt.tight_layout()
plt.show()

# Distribusi berat sebelum & sesudah makan (usable)
plt.figure(figsize=(12, 6))
sns.histplot(df_usable['Weight Before Eaten (g)'], bins=30, kde=True, color='blue', label='Before')
sns.histplot(df_usable['Weight After Eaten (g)'],  bins=30, kde=True, color='red',  label='After')
plt.legend()
plt.title('Distribusi Berat Sebelum & Sesudah Makan (524 usable samples)')
plt.xlabel('Berat (g)')
plt.ylabel('Jumlah Sampel')
plt.tight_layout()
plt.show()

# Distribusi leftover (gram)
plt.figure(figsize=(10, 5))
sns.histplot(df_usable['Weight Leftover (g)'], bins=30, kde=True, color='green')
plt.title('Distribusi Sisa Makanan (g)  -- 524 usable samples')
plt.xlabel('Berat Sisa (g)')
plt.ylabel('Jumlah Sampel')
plt.tight_layout()
plt.show()

# Distribusi consumption_ratio -- the actual training target on 524 filtered rows
plt.figure(figsize=(10, 5))
sns.histplot(df_usable['consumption_ratio'], bins=30, kde=True, color='teal')
plt.axvline(df_usable['consumption_ratio'].mean(), color='red', linestyle='--',
            label=f'Mean = {df_usable["consumption_ratio"].mean():.3f}')
plt.title('Distribusi Consumption Ratio (target variabel, 524 usable samples)')
plt.xlabel('Consumption Ratio  (Weight After / Weight Before)')
plt.ylabel('Jumlah Sampel')
plt.legend()
plt.tight_layout()
plt.show()

# Distribusi skor visual estimation
plt.figure(figsize=(8, 5))
sns.countplot(x='Visual Estimation by Observer (1-7)', data=df_usable,
              hue='Visual Estimation by Observer (1-7)', palette='viridis', legend=False)
plt.title('Distribusi Skor Visual Estimation (1-7)  -- 524 usable samples')
plt.xlabel('Skor')
plt.ylabel('Jumlah')
plt.tight_layout()
plt.show()

## 4. Analisis Gambar
Membaca gambar **raw** dari folder `data/raw/data_before` dan `data/raw/data_after` untuk menganalisis resolusi, ukuran file, dan kecerahan.

> Catatan: Training menggunakan gambar **segmented** (`data/segmented/`), bukan raw. Analisis ini hanya untuk memahami properti visual dataset.

In [ ]:
# Raw image folders (relative to project root)
folder_before = os.path.join('data', 'raw', 'data_before')
folder_after  = os.path.join('data', 'raw', 'data_after')


def find_file(base_folder, filename):
    for root, dirs, files in os.walk(base_folder):
        if filename in files:
            return os.path.join(root, filename)
    return None


def get_image_info(image_path):
    try:
        with Image.open(image_path) as img:
            width, height = img.size
            size_kb = os.path.getsize(image_path) / 1024
            arr = np.array(img)
            brightness = np.mean(arr)
            return width, height, size_kb, brightness
    except Exception:
        return None, None, None, None


image_stats = []
missing_before = []
missing_after = []

for idx, row in df.iterrows():
    before_path = find_file(folder_before, row['Image Before Eaten'])
    after_path  = find_file(folder_after,  row['Image After Eaten'])

    if before_path is None:
        missing_before.append(row['Image Before Eaten'])
    if after_path is None:
        missing_after.append(row['Image After Eaten'])

    for label, path in [('before', before_path), ('after', after_path)]:
        if path:
            width, height, size_kb, brightness = get_image_info(path)
            image_stats.append({
                'ID': row['ID'],
                'type': label,
                'file_name': os.path.basename(path),
                'width': width,
                'height': height,
                'size_kb': size_kb,
                'brightness': brightness
            })

print(f'Gambar BEFORE hilang: {len(missing_before)}')
print(f'Gambar AFTER hilang:  {len(missing_after)}')

img_df = pd.DataFrame(image_stats)

plt.figure(figsize=(10, 5))
sns.histplot(img_df['width'], bins=30, kde=True, color='purple')
plt.title('Distribusi Lebar Gambar (Raw)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(img_df['height'], bins=30, kde=True, color='orange')
plt.title('Distribusi Tinggi Gambar (Raw)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(img_df['brightness'], bins=30, kde=True, color='gray')
plt.title('Distribusi Kecerahan Gambar (Raw)')
plt.xlabel('Brightness (0-255)')
plt.tight_layout()
plt.show()

## 5. Contoh Gambar Before & After
Menampilkan satu sampel gambar before dan after untuk visualisasi perbandingan.

In [ ]:
sample_row = df_usable.sample(1, random_state=42).iloc[0]
before_img_path = find_file(folder_before, sample_row['Image Before Eaten'])
after_img_path  = find_file(folder_after,  sample_row['Image After Eaten'])

if before_img_path and after_img_path:
    before_img = Image.open(before_img_path)
    after_img  = Image.open(after_img_path)

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(before_img)
    plt.title('Before: ' + sample_row['Name of the food'])
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(after_img)
    plt.title(f"After  (ratio: {sample_row['consumption_ratio']:.3f}  |  leftover: {sample_row['Weight Leftover (g)']}g)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Sample image not found. Check that data/raw/ folders are populated.")

## 6. Korelasi Leftover vs Visual Estimation
Menghitung korelasi Pearson untuk mengetahui hubungan antara berat sisa makanan dengan skor penilaian visual.

In [ ]:
r, p = pearsonr(df_usable['Weight Leftover (g)'], df_usable['Visual Estimation by Observer (1-7)'])
print(f"Pearson r = {r:.4f}  (p = {p:.4e})")
print("Negative r expected: higher leftover -> lower visual score (score 1 = not eaten at all)")

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='Weight Leftover (g)',
    y='Visual Estimation by Observer (1-7)',
    data=df_usable, alpha=0.6
)
plt.title(f'Korelasi Sisa Makanan vs Skor Visual  (r = {r:.3f})  -- 524 usable samples')
plt.xlabel('Berat Sisa (g)')
plt.ylabel('Skor Visual (1-7)')
plt.tight_layout()
plt.show()

# Additional: consumption_ratio vs visual score
r2, p2 = pearsonr(df_usable['consumption_ratio'], df_usable['Visual Estimation by Observer (1-7)'])
print(f"\nconsumption_ratio vs visual score: Pearson r = {r2:.4f}  (p = {p2:.4e})")
print("Positive r expected: higher consumption ratio -> more was eaten -> higher visual score")

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='consumption_ratio',
    y='Visual Estimation by Observer (1-7)',
    data=df_usable, alpha=0.6
)
plt.title(f'Consumption Ratio vs Skor Visual  (r = {r2:.3f})')
plt.xlabel('Consumption Ratio  (Weight After / Weight Before)')
plt.ylabel('Skor Visual (1-7)')
plt.tight_layout()
plt.show()